In [ ]:
from __future__ import annotations

import argparse
import json
import math
import os
import time
from dataclasses import asdict, dataclass
from types import SimpleNamespace
from datetime import datetime
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [ ]:
# Config

@dataclass
class ToyMamba2Config:

    # Model dims
    d_model: int = 512
    d_head: int = 64
    d_state: int = 64
    expand: int = 2

    # Architecture
    n_layers: int = 17
    n_groups: int = 1
    conv_dim: int = 4
    block_len: int = 64

    # Vocab / seq
    vocab_size: int = 50257
    max_seq_len: int = 1024

    # Regularization
    dropout: float = 0.0

    # Misc
    tie_embeddings: bool = True
    norm_eps: float = 1e-6

    @classmethod
    def full(cls, **overrides) -> "ToyMamba2Config":
        defaults = dict(d_model=512, n_layers=17, d_head=64, d_state=64, expand=2)
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def half(cls, **overrides) -> "ToyMamba2Config":
        defaults = dict(d_model=384, n_layers=8, d_head=64, d_state=64, expand=2)
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def quarter(cls, **overrides) -> "ToyMamba2Config":
        defaults = dict(d_model=256, n_layers=4, d_head=64, d_state=64, expand=2)
        defaults.update(overrides)
        return cls(**defaults)

    @property
    def d_inner(self) -> int:
        return self.d_model * self.expand

    @property
    def n_heads(self) -> int:
        return self.d_inner // self.d_head

    def validate(self) -> None:
        assert self.d_model > 0
        assert self.d_head > 0
        assert self.d_state > 0
        assert self.expand > 0
        assert self.n_layers > 0
        assert self.n_groups > 0
        assert self.conv_dim >= 1
        assert self.block_len >= 1
        assert self.max_seq_len >= 1
        assert self.d_inner % self.d_head == 0, "d_inner must be divisible by d_head"
        assert self.n_heads % self.n_groups == 0, "n_heads must be divisible by n_groups"

In [ ]:
# Model components


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute in fp32 for stability, return in the input dtype for AMP friendliness.
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        y = x_float * rms * self.weight.float()
        return y.to(dtype)


def segsum(x: torch.Tensor) -> torch.Tensor:
   # Stable segment-sum helper for a lower-triangular 1-semiseparable mask

    T = x.size(-1)
    cumsum = x.cumsum(dim=-1)
    diff = cumsum[..., :, None] - cumsum[..., None, :]
    mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
    return diff.masked_fill(~mask, -float("inf"))


def ssd(
    X: torch.Tensor,      # [B, T, H, P]
    A: torch.Tensor,      # [B, T, H], log-decay values, usually negative
    B: torch.Tensor,      # [B, T, H, N]
    C: torch.Tensor,      # [B, T, H, N]
    block_len: int = 64,
    initial_states: Optional[torch.Tensor] = None,  # [B, 1, H, P, N]
) -> tuple[torch.Tensor, torch.Tensor]:
    # Block-decomposed SSD forward pass.

    #Returns:
    #   Y: [B, T, H, P]
    #   final_state: [B, H, P, N]

    batch, T_orig, H, P = X.shape
    N = B.shape[-1]
    Q = block_len

    assert A.shape == (batch, T_orig, H)
    assert B.shape == (batch, T_orig, H, N)
    assert C.shape == (batch, T_orig, H, N)

    pad = (Q - T_orig % Q) % Q
    if pad > 0:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
        B = F.pad(B, (0, 0, 0, 0, 0, pad))
        C = F.pad(C, (0, 0, 0, 0, 0, pad))

    T = T_orig + pad
    n_chunks = T // Q

    X = X.reshape(batch, n_chunks, Q, H, P)
    A = A.reshape(batch, n_chunks, Q, H)
    B = B.reshape(batch, n_chunks, Q, H, N)
    C = C.reshape(batch, n_chunks, Q, H, N)

    # [B, H, C, Q]
    A = A.permute(0, 3, 1, 2).contiguous()
    A_cumsum = A.cumsum(dim=-1)

    # Diagonal blocks: intra-chunk quadratic part.
    L = segsum(A).exp()  # [B, H, C, Q, Q]
    G = torch.einsum("bclhn,bcshn->bhcls", C, B)
    Y_diag = torch.einsum("bhcls,bcshp->bclhp", L * G, X)

    # Right factors: input positions to per-chunk state.
    decay_states = (A_cumsum[..., -1:] - A_cumsum).exp()  # [B, H, C, Q]
    chunk_states = torch.einsum("bclhn,bhcl,bclhp->bchpn", B, decay_states, X)

    # Center factors: chunk-to-chunk recurrence/scan.
    if initial_states is None:
        initial_states = chunk_states.new_zeros(batch, 1, H, P, N)
    else:
        assert initial_states.shape == (batch, 1, H, P, N)

    states_with_initial = torch.cat([initial_states, chunk_states], dim=1)

    # A_chunk_total[:, :, 0] corresponds to the initial state boundary.
    A_chunk_total = F.pad(A_cumsum[..., -1], (1, 0))  # [B, H, C + 1]
    decay_chunk = segsum(A_chunk_total).exp()         # [B, H, C + 1, C + 1]
    scanned_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states_with_initial)

    states = scanned_states[:, :-1]       # [B, C, H, P, N]
    final_state = scanned_states[:, -1]   # [B, H, P, N]

    # 4. Left factors: per-chunk state to outputs.
    state_decay_out = A_cumsum.exp()  # [B, H, C, Q]
    Y_off = torch.einsum("bclhn,bchpn,bhcl->bclhp", C, states, state_decay_out)

    Y = (Y_diag + Y_off).reshape(batch, T, H, P)
    if pad > 0:
        Y = Y[:, :T_orig]

    return Y, final_state


@torch.no_grad()
def ssd_recurrent_naive(
    X: torch.Tensor,
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
) -> torch.Tensor:
    # Slow recurrent SSD reference implementation. Use only for correctness tests

    batch, T, H, P = X.shape
    N = B.shape[-1]
    state = X.new_zeros(batch, H, P, N)
    outputs = []

    for t in range(T):
        decay = torch.exp(A[:, t]).view(batch, H, 1, 1)
        state = decay * state + torch.einsum("bhn,bhp->bhpn", B[:, t], X[:, t])
        y_t = torch.einsum("bhn,bhpn->bhp", C[:, t], state)
        outputs.append(y_t)

    return torch.stack(outputs, dim=1)


def test_ssd_correctness(device: torch.device, dtype: torch.dtype = torch.float32) -> None:
     #Verifies that block SSD matches the naive recurrence for several sequence lengths, including lengths not divisible by block_len

    torch.manual_seed(0)
    test_cases = [
        dict(T=1, block_len=4),
        dict(T=7, block_len=4),
        dict(T=16, block_len=8),
        dict(T=17, block_len=8),
        dict(T=33, block_len=16),
    ]

    for case in test_cases:
        batch, T, H, P, N = 2, case["T"], 3, 5, 7
        block_len = case["block_len"]

        X = torch.randn(batch, T, H, P, device=device, dtype=dtype)
        # Keep A negative and not too large in magnitude to avoid numerical extremes.
        A = -0.1 * torch.rand(batch, T, H, device=device, dtype=dtype)
        B_ = torch.randn(batch, T, H, N, device=device, dtype=dtype)
        C_ = torch.randn(batch, T, H, N, device=device, dtype=dtype)

        Y_fast, _ = ssd(X, A, B_, C_, block_len=block_len)
        Y_slow = ssd_recurrent_naive(X, A, B_, C_)
        max_err = (Y_fast.float() - Y_slow.float()).abs().max().item()

        tolerance = 2e-4 if dtype == torch.float32 else 5e-2
        if max_err > tolerance:
            raise AssertionError(
                f"SSD correctness failed for T={T}, block_len={block_len}: "
                f"max_err={max_err:.6g}, tolerance={tolerance:.6g}"
            )

    print("SSD correctness test passed.")


@torch.no_grad()
def test_recurrent_matches_full_forward(device: torch.device) -> None:
    # Verifies that recurrent prefill gives approximately the same logits as the full-sequence forward path.

    torch.manual_seed(123)
    cfg = ToyMamba2Config.quarter(max_seq_len=32, block_len=8, dropout=0.0)
    model = ToyMamba2(cfg).to(device)
    model.eval()

    input_ids = torch.randint(0, cfg.vocab_size, (2, 19), device=device)
    logits_full, _ = model(input_ids)

    caches = model.allocate_cache(
        batch_size=input_ids.size(0),
        device=device,
        dtype=model.tok_emb.weight.dtype,
    )
    logits_step = None
    for t in range(input_ids.size(1)):
        logits_step, caches = model.step(input_ids[:, t], caches)

    assert logits_step is not None
    max_err = (logits_full[:, -1].float() - logits_step.float()).abs().max().item()
    tolerance = 5e-4
    if max_err > tolerance:
        raise AssertionError(
            f"Recurrent/full-forward mismatch: max_err={max_err:.6g}, "
            f"tolerance={tolerance:.6g}"
        )

    print("Recurrent step-by-step prefill matches full forward test passed.")


@torch.no_grad()
def test_optimized_prefill_matches_full_and_step(device: torch.device) -> None:
    # Verifies optimized parallel prefill cache extraction

    #Test 1: prefill(prompt) logits match normal full forward(prompt) logits
    # Test 2: prefill(prompt) + step(next_token) matches full forward(prompt + next_token) on the final token logits

    #Trying to catch off-by-one errors in conv_state and SSM final_state extraction

    torch.manual_seed(456)
    cfg = ToyMamba2Config.quarter(max_seq_len=64, block_len=8, dropout=0.0)
    model = ToyMamba2(cfg).to(device)
    model.eval()

    prompt = torch.randint(0, cfg.vocab_size, (2, 19), device=device)
    next_token = torch.randint(0, cfg.vocab_size, (2,), device=device)
    prompt_plus_one = torch.cat([prompt, next_token[:, None]], dim=1)

    logits_full_prompt, _ = model(prompt)
    logits_prefill, caches = model.prefill(prompt)

    err_prefill = (logits_full_prompt.float() - logits_prefill.float()).abs().max().item()
    tolerance_prefill = 5e-4
    if err_prefill > tolerance_prefill:
        raise AssertionError(
            f"Optimized prefill logits mismatch: max_err={err_prefill:.6g}, "
            f"tolerance={tolerance_prefill:.6g}"
        )

    logits_full_plus_one, _ = model(prompt_plus_one)
    logits_next_step, _ = model.step(next_token, caches)

    err_next = (logits_full_plus_one[:, -1].float() - logits_next_step.float()).abs().max().item()
    tolerance_next = 5e-4
    if err_next > tolerance_next:
        raise AssertionError(
            f"Optimized prefill + step mismatch: max_err={err_next:.6g}, "
            f"tolerance={tolerance_next:.6g}"
        )

    print("Optimized parallel prefill cache test passed.")


@dataclass
class Mamba2LayerCache:
    conv_state: torch.Tensor  # [B, conv_channels, conv_kernel - 1]
    ssm_state: torch.Tensor   # [B, H, P, N]


class Mamba2Block(nn.Module):
    def __init__(self, cfg: ToyMamba2Config):
        super().__init__()
        cfg.validate()

        d = cfg.d_model
        E = cfg.d_inner
        H = cfg.n_heads
        N = cfg.d_state
        P = cfg.d_head
        G = cfg.n_groups

        self.H = H
        self.P = P
        self.N = N
        self.G = G
        self.E = E
        self.block_len = cfg.block_len
        self.dropout_p = cfg.dropout

        conv_channels = E + 2 * G * N
        proj_dim = E + conv_channels + H

        self.in_proj = nn.Linear(d, proj_dim, bias=False)

        self.conv1d = nn.Conv1d(
            conv_channels,
            conv_channels,
            kernel_size=cfg.conv_dim,
            padding=cfg.conv_dim - 1,
            groups=conv_channels,
            bias=True,
        )

        # Scalar-identity A per head. A = -exp(A_log).
        self.A_log = nn.Parameter(torch.log(torch.empty(H).uniform_(1, 16)))

        # Inverse softplus initialization for dt in approximately [0.001, 0.1].
        dt_init = torch.exp(
            torch.rand(H) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)
        ).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt_init + torch.log(-torch.expm1(-dt_init)))

        # Per-head skip/feed-through.
        self.D = nn.Parameter(torch.ones(H))

        # Mamba-2-style order:
        #     y = y * silu(z)
        #     y = norm(y)
        #     out = out_proj(y)
        self.norm = RMSNorm(E, eps=cfg.norm_eps)
        self.dropout = nn.Dropout(cfg.dropout)
        self.out_proj = nn.Linear(E, d, bias=False)

        self.conv_channels = conv_channels
        self.conv_kernel_size = cfg.conv_dim
        self.proj_splits = [E, conv_channels, H]
        self.conv_splits = [E, G * N, G * N]

    def allocate_cache(
        self,
        batch_size: int,
        device: torch.device,
        dtype: torch.dtype,
    ) -> Mamba2LayerCache:
        # Allocate recurrent inference cache for one Mamba block

        return Mamba2LayerCache(
            conv_state=torch.zeros(
                batch_size,
                self.conv_channels,
                self.conv_kernel_size - 1,
                device=device,
                dtype=dtype,
            ),
            ssm_state=torch.zeros(
                batch_size,
                self.H,
                self.P,
                self.N,
                device=device,
                dtype=dtype,
            ),
        )

    def _project_and_conv_step(
        self,
        u_t: torch.Tensor,  # [B, d_model]
        cache: Mamba2LayerCache,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, Mamba2LayerCache]:

        batch = u_t.size(0)
        H, P, N, G = self.H, self.P, self.N, self.G

        z, xBC, dt = self.in_proj(u_t).split(self.proj_splits, dim=-1)

        # Manual one-step causal depthwise conv.
        # Training path uses Conv1d with padding=K-1 and then crops to [:T].
        # For one-step inference, the equivalent input window is the cached K-1 previous xBC vectors plus the current xBC vector.
        xBC_ch = xBC.unsqueeze(-1)  # [B, conv_channels, 1]
        if self.conv_kernel_size > 1:
            conv_window = torch.cat([cache.conv_state, xBC_ch], dim=-1)
        else:
            conv_window = xBC_ch

        # Conv1d is depthwise: weight [conv_channels, 1, K].
        weight = self.conv1d.weight[:, 0, :].to(conv_window.dtype)  # [conv_channels, K]
        xBC_conv = (conv_window * weight.unsqueeze(0)).sum(dim=-1)
        if self.conv1d.bias is not None:
            xBC_conv = xBC_conv + self.conv1d.bias.to(xBC_conv.dtype)
        xBC_conv = F.silu(xBC_conv)

        if self.conv_kernel_size > 1:
            cache.conv_state = conv_window[:, :, 1:].contiguous()

        x, b_flat, c_flat = xBC_conv.split(self.conv_splits, dim=-1)
        x = x.reshape(batch, H, P)

        b = b_flat.reshape(batch, G, N)
        c = c_flat.reshape(batch, G, N)
        if G != H:
            repeats = H // G
            b = b.repeat_interleave(repeats, dim=1)
            c = c.repeat_interleave(repeats, dim=1)

        return z, x, b, c, dt, cache

    def step(
        self,
        u_t: torch.Tensor,  # [B, d_model]
        cache: Mamba2LayerCache,
    ) -> tuple[torch.Tensor, Mamba2LayerCache]:
        # Recurrent one-token inference step.
        # equivalent to the full-sequence path under this toy SSD parameterization, up to numerical tolerance

        batch = u_t.size(0)
        H = self.H

        z, x, b, c, dt, cache = self._project_and_conv_step(u_t, cache)

        A = -torch.exp(self.A_log.float()).to(dt.dtype)  # [H]
        dt_pos = F.softplus(dt + self.dt_bias.to(dt.dtype))
        a = dt_pos * A.view(1, H)                       # [B, H]
        decay = torch.exp(a).view(batch, H, 1, 1)

        cache.ssm_state = (
            decay * cache.ssm_state
            + torch.einsum("bhn,bhp->bhpn", b, x)
        )

        y = torch.einsum("bhn,bhpn->bhp", c, cache.ssm_state)
        y = y + self.D.to(y.dtype).view(1, H, 1) * x
        y = y.reshape(batch, self.E)

        y = y * F.silu(z)
        y = self.norm(y)
        y = self.out_proj(y)
        return y, cache

    def _forward_impl(
        self,
        u: torch.Tensor,
        return_cache: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, Mamba2LayerCache]:
        # Full-sequence block forward.

        batch, T, _ = u.shape
        H, P, N, G, E = self.H, self.P, self.N, self.G, self.E

        z, raw_xBC, dt = self.in_proj(u).split(self.proj_splits, dim=-1)

        # Save the raw pre-conv xBC tail for recurrent conv_state.
        # The recurrent conv step needs the last K-1 raw projected xBC vectors.
        if return_cache and self.conv_kernel_size > 1:
            tail_len = self.conv_kernel_size - 1
            if T >= tail_len:
                conv_state = raw_xBC[:, -tail_len:, :].transpose(1, 2).contiguous()
            else:
                # Left-pad with zeros if the prompt is shorter than K-1.
                pad_len = tail_len - T
                zeros = raw_xBC.new_zeros(batch, pad_len, self.conv_channels)
                conv_state = torch.cat([zeros, raw_xBC], dim=1).transpose(1, 2).contiguous()
        elif return_cache:
            conv_state = raw_xBC.new_zeros(batch, self.conv_channels, 0)

        # Causal depthwise conv. Conv1d padding adds left and right padding; crop to T.
        xBC = self.conv1d(raw_xBC.transpose(1, 2))[..., :T].transpose(1, 2)
        xBC = F.silu(xBC)
        x, b_flat, c_flat = xBC.split(self.conv_splits, dim=-1)

        x = x.reshape(batch, T, H, P)

        b = b_flat.reshape(batch, T, G, N)
        c = c_flat.reshape(batch, T, G, N)

        if G != H:
            repeats = H // G
            b = b.repeat_interleave(repeats, dim=2)
            c = c.repeat_interleave(repeats, dim=2)

        # Toy SSD convention: pass log-decay A into ssd(); ssd() exponentiates it.
        A = -torch.exp(self.A_log.float()).to(dt.dtype)  # [H]
        dt_pos = F.softplus(dt + self.dt_bias.to(dt.dtype))
        a = dt_pos * A.view(1, 1, H)                     # [B, T, H], negative

        y, final_state = ssd(x, a, b, c, self.block_len)

        # D skip connection.
        y = y + self.D.to(y.dtype).view(1, 1, H, 1) * x

        y = y.reshape(batch, T, E)

        y = y * F.silu(z)
        y = self.norm(y)
        y = self.dropout(y)
        out = self.out_proj(y)

        if return_cache:
            cache = Mamba2LayerCache(
                conv_state=conv_state,
                ssm_state=final_state.contiguous(),
            )
            return out, cache
        return out

    def forward_with_cache(self, u: torch.Tensor) -> tuple[torch.Tensor, Mamba2LayerCache]:
        #Parallel/full-sequence prefill for one block.
        #Returns the full block output and a recurrent cache that can be used for the next token

        out, cache = self._forward_impl(u, return_cache=True)
        return out, cache

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        return self._forward_impl(u, return_cache=False)


class ToyMamba2(nn.Module):
    def __init__(self, cfg: ToyMamba2Config):
        super().__init__()
        cfg.validate()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "norm": RMSNorm(cfg.d_model, cfg.norm_eps),
                        "mamba": Mamba2Block(cfg),
                    }
                )
                for _ in range(cfg.n_layers)
            ]
        )
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        self.apply(self._init_weights)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def allocate_cache(
        self,
        batch_size: int,
        device: torch.device,
        dtype: torch.dtype,
    ) -> list[Mamba2LayerCache]:
        # Allocate recurrent inference caches for all layers
        return [
            layer["mamba"].allocate_cache(batch_size, device, dtype)
            for layer in self.layers
        ]

    @torch.no_grad()
    def prefill(
        self,
        input_ids: torch.Tensor,
    ) -> tuple[torch.Tensor, list[Mamba2LayerCache]]:
        # Kinda optimized prompt prefill.

        # Runs the prompt through the normal full-sequence SSD path, which is parallel over sequence length, while extracting recurrent caches for all
        #layers. Use before recurrent token-by-token decoding.
        x = self.tok_emb(input_ids)
        x = self.drop(x)
        caches: list[Mamba2LayerCache] = []

        for layer in self.layers:
            residual = x
            x_norm = layer["norm"](x)
            y, cache = layer["mamba"].forward_with_cache(x_norm)
            x = residual + y
            caches.append(cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, caches

    @torch.no_grad()
    def step(
        self,
        input_id_t: torch.Tensor,  # [B]
        caches: list[Mamba2LayerCache],
    ) -> tuple[torch.Tensor, list[Mamba2LayerCache]]:
        # One autoregressive recurrent step.

        x = self.tok_emb(input_id_t)  # [B, d_model]
        new_caches: list[Mamba2LayerCache] = []

        for layer, cache in zip(self.layers, caches):
            residual = x
            x_norm = layer["norm"](x)
            y, new_cache = layer["mamba"].step(x_norm, cache)
            x = residual + y
            new_caches.append(new_cache)

        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits, new_caches

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
        x = self.tok_emb(input_ids)
        x = self.drop(x)

        for layer in self.layers:
            x = x + layer["mamba"](layer["norm"](x))

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )
        return logits, loss

    @torch.no_grad()
    def generate_full_prefix(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
    ) -> torch.Tensor:
        #Simple generation by full-prefix recomputation
        # Potentially correct but not optimized

        self.eval()
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.cfg.max_seq_len :]
            logits, _ = self.forward(idx)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k > 0:
                values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_id], dim=1)

        return input_ids

    @torch.no_grad()
    def generate_recurrent(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        optimized_prefill: bool = True,
    ) -> torch.Tensor:
        self.eval()
        batch_size = input_ids.size(0)
        device = input_ids.device
        cache_dtype = self.tok_emb.weight.dtype

        if optimized_prefill:
            prefill_logits, caches = self.prefill(input_ids)
            logits = prefill_logits[:, -1, :]
        else:
            caches = self.allocate_cache(batch_size, device, cache_dtype)
            logits = None
            for t in range(input_ids.size(1)):
                logits, caches = self.step(input_ids[:, t], caches)
            assert logits is not None

        out = input_ids

        for _ in range(max_new_tokens):
            next_logits = logits / max(temperature, 1e-8)
            if top_k > 0:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits = next_logits.masked_fill(next_logits < values[:, [-1]], -float("inf"))

            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(-1)
            out = torch.cat([out, next_id[:, None]], dim=1)
            logits, caches = self.step(next_id, caches)

        return out

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 128,
        temperature: float = 0.8,
        top_k: int = 40,
        use_recurrent: bool = True,
        optimized_prefill: bool = True,
    ) -> torch.Tensor:
        if use_recurrent:
            return self.generate_recurrent(
                input_ids,
                max_new_tokens,
                temperature,
                top_k,
                optimized_prefill=optimized_prefill,
            )
        return self.generate_full_prefix(input_ids, max_new_tokens, temperature, top_k)

    def num_parameters(self, only_trainable: bool = True) -> int:
        return sum(
            p.numel()
            for p in self.parameters()
            if (not only_trainable or p.requires_grad)
        )

In [ ]:
# Data Preprocessing and Handling


class PackedTokenDataset(Dataset):
    # Concatenates token ids and slices them into fixed-length windows

    def __init__(self, token_ids: list[int], seq_len: int):
        if len(token_ids) < seq_len + 1:
            raise ValueError(
                f"Need at least seq_len + 1 tokens, got {len(token_ids)} tokens "
                f"for seq_len={seq_len}."
            )
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len
        self.n_samples = (len(self.data) - 1) // seq_len

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        start = idx * self.seq_len
        chunk = self.data[start : start + self.seq_len + 1]
        return chunk[:-1], chunk[1:]


def _row_to_text(row: dict) -> str:
    for key in ("text", "content", "story"):
        value = row.get(key)
        if isinstance(value, str):
            return value
    return ""


def load_tokens(
    dataset_name: str,
    tokenizer,
    split: str = "train",
    max_tokens: Optional[int] = None,
) -> list[int]:
    """Returns a flat list of token ids for a file or HF dataset split."""
    from datasets import load_dataset

    if dataset_name.startswith("file:"):
        path = dataset_name[5:]
        with open(path, encoding="utf-8") as f:
            text = f.read()
        tokens = tokenizer.encode(text)
        return tokens[:max_tokens] if max_tokens else tokens

    ds_configs = {
        "wikitext-2": ("wikitext", "wikitext-2-raw-v1", False),
        "wikitext-103": ("wikitext", "wikitext-103-raw-v1", False),
        "tinystories": ("roneneldan/TinyStories", None, False),
        "openwebtext": ("Skylion007/openwebtext", None, True),
        "fineweb-edu": ("HuggingFaceFW/fineweb-edu-score-2", "sample-10BT", True),
        "slimpajama": ("cerebras/SlimPajama-627B", None, True),
        "c4": ("allenai/c4", "en", True),
    }

    if dataset_name not in ds_configs:
        raise ValueError(
            f"Unknown dataset '{dataset_name}'. Available: {', '.join(ds_configs)}, "
            "or file:<path>."
        )

    ds_name, ds_config, streaming = ds_configs[dataset_name]
    tokens: list[int] = []
    limit = max_tokens or 500_000_000

    if streaming:
        ds = load_dataset(ds_name, ds_config, split=split, streaming=True)
        for row in ds:
            text = _row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
                if len(tokens) >= limit:
                    tokens = tokens[:limit]
                    break
    else:
        ds = load_dataset(ds_name, ds_config, split=split)
        for row in ds:
            text = _row_to_text(row)
            if text.strip():
                tokens.extend(tokenizer.encode(text))
                if len(tokens) >= limit:
                    tokens = tokens[:limit]
                    break

    return tokens


def make_data_splits(
    dataset_name: str,
    tokenizer,
    max_tokens: Optional[int],
    val_monitor_fraction: float,
    val_ablation_fraction: float,
    test_replication_fraction: float,
    val_monitor_max_tokens: Optional[int],
    val_ablation_max_tokens: Optional[int],
    test_replication_max_tokens: Optional[int],
) -> tuple[dict[str, list[int]], dict]:
    # Creates train / val_monitor / val_ablation / test_replication token splits.

    split_info = {
        "dataset": dataset_name,
        "split_policy": None,
        "fractions": {
            "val_monitor": val_monitor_fraction,
            "val_ablation": val_ablation_fraction,
            "test_replication": test_replication_fraction,
        },
    }

    def cap_tokens(tokens: list[int], cap: Optional[int]) -> list[int]:
        return tokens[:cap] if cap is not None else tokens

    if dataset_name in {"wikitext-2", "wikitext-103"}:
        train_all = load_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        val_monitor = load_tokens(dataset_name, tokenizer, split="validation", max_tokens=val_monitor_max_tokens)
        test_replication = load_tokens(dataset_name, tokenizer, split="test", max_tokens=test_replication_max_tokens)

        n_ablation = max(int(len(train_all) * val_ablation_fraction), 1)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)
        train = train_all[:-n_ablation]
        val_ablation = train_all[-n_ablation:]

        split_info["split_policy"] = "official validation/test; val_ablation carved from train tail"
    else:
        all_tokens = load_tokens(dataset_name, tokenizer, split="train", max_tokens=max_tokens)
        n_total = len(all_tokens)

        n_monitor = max(int(n_total * val_monitor_fraction), 1)
        n_ablation = max(int(n_total * val_ablation_fraction), 1)
        n_test = max(int(n_total * test_replication_fraction), 1)

        if val_monitor_max_tokens is not None:
            n_monitor = min(n_monitor, val_monitor_max_tokens)
        if val_ablation_max_tokens is not None:
            n_ablation = min(n_ablation, val_ablation_max_tokens)
        if test_replication_max_tokens is not None:
            n_test = min(n_test, test_replication_max_tokens)

        heldout_total = n_monitor + n_ablation + n_test
        if n_total - heldout_total < 2:
            raise ValueError(
                "Training split is too small after held-out splits. "
                "Increase max_tokens or reduce split fractions."
            )

        train_end = n_total - heldout_total
        monitor_end = train_end + n_monitor
        ablation_end = monitor_end + n_ablation

        train = all_tokens[:train_end]
        val_monitor = all_tokens[train_end:monitor_end]
        val_ablation = all_tokens[monitor_end:ablation_end]
        test_replication = all_tokens[ablation_end:]

        split_info["split_policy"] = "tail holdout from train token stream"
        split_info["n_total_loaded_tokens"] = n_total

    splits = {
        "train": train,
        "val_monitor": cap_tokens(val_monitor, val_monitor_max_tokens),
        "val_ablation": cap_tokens(val_ablation, val_ablation_max_tokens),
        "test_replication": cap_tokens(test_replication, test_replication_max_tokens),
    }

    split_info["n_tokens"] = {name: len(tokens) for name, tokens in splits.items()}
    split_info["heldout_names"] = ["val_monitor", "val_ablation", "test_replication"]
    return splits, split_info


def verify_splits(
    splits: dict[str, list[int]],
    seq_len: int,
) -> dict:
    # Sanity-checks all data splits.

    # Because every split is turned into its own PackedTokenDataset, no packed training window can cross from one split into another.

    required = ["train", "val_monitor", "val_ablation", "test_replication"]
    for name in required:
        if name not in splits:
            raise KeyError(f"Missing split: {name}")
        if len(splits[name]) < seq_len + 1:
            raise ValueError(
                f"Split '{name}' too small: {len(splits[name])} tokens for seq_len={seq_len}."
            )

    checks = {}
    for name, tokens in splits.items():
        checks[name] = {
            "tokens": len(tokens),
            "samples": (len(tokens) - 1) // seq_len,
        }

    checks["windows_cross_boundary"] = False
    checks["note"] = (
        "PackedTokenDataset is built separately for each split, so windows do not cross boundaries. "
        "This does not prove semantic deduplication."
    )
    return checks


def verify_split(
    train_tokens: list[int],
    val_tokens: list[int],
    seq_len: int,
    strict_boundary_check: bool = True,
) -> dict:
    # Sanity-checks train/validation split.

    # This verifies:
      # both splits are large enough to produce at least one training example
      # the train and validation lists are distinct Python objects
      # no sequence window crosses the split boundary

    # It does not prove that the text content is semantically deduplicated.

    if len(train_tokens) < seq_len + 1:
        raise ValueError(
            f"Train split too small: {len(train_tokens)} tokens for seq_len={seq_len}."
        )
    if len(val_tokens) < seq_len + 1:
        raise ValueError(
            f"Validation split too small: {len(val_tokens)} tokens for seq_len={seq_len}."
        )

    checks = {
        "train_tokens": len(train_tokens),
        "val_tokens": len(val_tokens),
        "train_samples": (len(train_tokens) - 1) // seq_len,
        "val_samples": (len(val_tokens) - 1) // seq_len,
        "same_object": train_tokens is val_tokens,
        "windows_cross_boundary": False,
    }

    if checks["same_object"]:
        raise AssertionError("Train and validation token lists are the same object.")

    # Because PackedTokenDataset is built separately for train and val, no window can cross.
    # This flag is included explicitly because that was the main split concern.
    if strict_boundary_check and checks["windows_cross_boundary"]:
        raise AssertionError("A packed window crosses the train/validation boundary.")

    return checks

In [ ]:
# Training helpers


class TrainingLogger:
    def __init__(self, log_path: str, run_config: dict):
        self.log_path = Path(log_path)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(self.log_path, "w", encoding="utf-8")
        self.write({"type": "run_config", "timestamp": datetime.now().isoformat(), **run_config})

    def write(self, payload: dict) -> None:
        self.file.write(json.dumps(payload, default=str) + "\n")
        self.file.flush()

    def log_step(self, metrics: dict) -> None:
        self.write({"type": "step", "timestamp": datetime.now().isoformat(), **metrics})

    def log_eval(self, metrics: dict) -> None:
        self.write({"type": "eval", "timestamp": datetime.now().isoformat(), **metrics})

    def log_final(self, metrics: dict) -> None:
        self.write({"type": "final", "timestamp": datetime.now().isoformat(), **metrics})

    def close(self) -> None:
        self.file.close()


def cosine_with_warmup(step: int, warmup: int, total: int) -> float:
    if step < warmup:
        return step / max(warmup, 1)
    progress = (step - warmup) / max(total - warmup, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def gpu_stats(device: torch.device) -> dict:
    if device.type != "cuda":
        return {}
    return {
        "gpu_mem_allocated_gb": round(torch.cuda.memory_allocated(device) / 1e9, 3),
        "gpu_mem_reserved_gb": round(torch.cuda.memory_reserved(device) / 1e9, 3),
        "gpu_mem_peak_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 3),
    }


def estimate_mamba2_flops_per_optimizer_step(cfg: ToyMamba2Config, effective_batch_size: int) -> dict:
    # Rough estimate only

    T = cfg.max_seq_len
    Q = cfg.block_len
    d = cfg.d_model
    E = cfg.d_inner
    H = cfg.n_heads
    P = cfg.d_head
    N = cfg.d_state
    L = cfg.n_layers
    G = cfg.n_groups

    flops_ssd_per_head = T * Q * N + T * Q * P + 2 * T * N * P
    flops_ssd = L * H * flops_ssd_per_head

    proj_in_dim = E + (E + 2 * G * N) + H
    flops_in_proj = L * 2 * T * d * proj_in_dim
    flops_out_proj = L * 2 * T * E * d

    # Embedding lookup is not a matmul. LM head is.
    flops_lm_head = 2 * T * d * cfg.vocab_size

    flops_fwd_per_sample = flops_ssd + flops_in_proj + flops_out_proj + flops_lm_head
    flops_per_opt_step = 3 * flops_fwd_per_sample * effective_batch_size

    return {
        "estimated_flops_fwd_per_sample": flops_fwd_per_sample,
        "estimated_flops_per_optimizer_step": flops_per_opt_step,
        "breakdown": {
            "ssd": flops_ssd,
            "in_projection": flops_in_proj,
            "out_projection": flops_out_proj,
            "lm_head": flops_lm_head,
        },
    }


def make_optimizer(model: nn.Module, lr: float, weight_decay: float) -> torch.optim.Optimizer:
    decay = []
    no_decay = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        name_lower = name.lower()
        if (
            name.endswith("bias")
            or "norm" in name_lower
            or "dt_bias" in name
            or "A_log" in name
            or name.endswith(".D")
        ):
            no_decay.append(param)
        else:
            decay.append(param)

    fused = torch.cuda.is_available()
    return torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=lr,
        betas=(0.9, 0.95),
        fused=fused,
    )


@torch.no_grad()
def evaluate(
    model: ToyMamba2,
    loader: DataLoader,
    device: torch.device,
    amp_dtype: torch.dtype,
    max_batches: int,
    use_amp: bool,
) -> dict:
    model.eval()
    losses = []
    n_tokens = 0
    t0 = time.time()

    for batch_idx, (x, y) in enumerate(loader):
        if batch_idx >= max_batches:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
            _, loss = model(x, y)

        losses.append(float(loss.item()))
        n_tokens += x.numel()

    if not losses:
        raise RuntimeError("Validation loader produced no batches.")

    val_loss = sum(losses) / len(losses)
    return {
        "val_loss": round(val_loss, 5),
        "val_ppl": round(math.exp(min(val_loss, 20)), 3),
        "val_batches": len(losses),
        "val_tokens": n_tokens,
        "val_time_sec": round(time.time() - t0, 2),
    }


def save_checkpoint(
    path: str | Path,
    model: ToyMamba2,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LambdaLR,
    cfg: ToyMamba2Config,
    optimizer_step: int,
    micro_step: int,
    best_val_loss: float,
    extra: Optional[dict] = None,
) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "optimizer_step": optimizer_step,
        "micro_step": micro_step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "config": asdict(cfg),
        "best_val_loss": best_val_loss,
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)
    print(f"saved checkpoint: {path}")


@dataclass
class EarlyStoppingState:
    patience: int
    min_delta: float
    warmup_evals: int = 0
    best_val_loss: float = float("inf")
    best_step: int = 0
    bad_evals: int = 0
    eval_count: int = 0

    def update(self, val_loss: float, step: int) -> tuple[bool, bool]:
        # Returns improved: whether validation loss improved enough
        #      should_stop: whether early stopping is triggered

        self.eval_count += 1

        improved = val_loss < (self.best_val_loss - self.min_delta)
        if improved:
            self.best_val_loss = val_loss
            self.best_step = step
            self.bad_evals = 0
        else:
            if self.eval_count > self.warmup_evals:
                self.bad_evals += 1

        should_stop = self.bad_evals >= self.patience
        return improved, should_stop


def is_model_trained_enough(
    current_val_loss: float,
    best_val_loss: float,
    recent_val_losses: list[float],
    min_relative_improvement: float = 0.002,
) -> dict:
    # metric-based training status helper.

    # This does not prove that the model is 'done' in an absolute sense. It checks whether validation loss has plateaued over the recent evaluation window.

    if len(recent_val_losses) < 4:
        return {"status": "too_early", "reason": "Need at least 4 validation points."}

    first = recent_val_losses[0]
    last = recent_val_losses[-1]
    rel_improvement = (first - last) / max(abs(first), 1e-8)

    if current_val_loss > best_val_loss * 1.01:
        return {
            "status": "overfitting_or_regressing",
            "reason": "Current validation loss is more than 1% worse than best validation loss.",
            "relative_improvement_window": rel_improvement,
        }

    if rel_improvement < min_relative_improvement:
        return {
            "status": "plateaued",
            "reason": "Recent validation loss improvement is below threshold.",
            "relative_improvement_window": rel_improvement,
        }

    return {
        "status": "still_learning",
        "reason": "Validation loss is still improving meaningfully.",
        "relative_improvement_window": rel_improvement,
    }

In [ ]:
# Main training loop


def train(args: argparse.Namespace) -> None:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        use_amp = True
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        # Safer default for correctness; AMP on MPS can be version-dependent.
        amp_dtype = torch.float32
        use_amp = False
    else:
        device = torch.device("cpu")
        amp_dtype = torch.float32
        use_amp = False

    print(f"Device: {device} | amp_dtype: {amp_dtype} | use_amp: {use_amp}")

    if args.test_ssd:
        test_ssd_correctness(device=torch.device("cpu"), dtype=torch.float32)

    if args.test_recurrent:
        # CPU fp32 gives the cleanest deterministic check
        test_recurrent_matches_full_forward(device=torch.device("cpu"))
        test_optimized_prefill_matches_full_and_step(device=torch.device("cpu"))

    factory = {
        "full": ToyMamba2Config.full,
        "half": ToyMamba2Config.half,
        "quarter": ToyMamba2Config.quarter,
    }[args.model_size]

    cfg = factory(
        max_seq_len=args.seq_len,
        block_len=args.block_len,
        dropout=args.dropout,
    )
    cfg.validate()

    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("gpt2", model_max_length=1_000_000)

    print("Loading and splitting data...")
    token_splits, split_info = make_data_splits(
        dataset_name=args.dataset,
        tokenizer=tokenizer,
        max_tokens=args.max_tokens,
        val_monitor_fraction=args.val_monitor_fraction,
        val_ablation_fraction=args.val_ablation_fraction,
        test_replication_fraction=args.test_replication_fraction,
        val_monitor_max_tokens=args.val_monitor_max_tokens,
        val_ablation_max_tokens=args.val_ablation_max_tokens,
        test_replication_max_tokens=args.test_replication_max_tokens,
    )
    split_checks = verify_splits(token_splits, cfg.max_seq_len)

    train_tokens = token_splits["train"]
    val_monitor_tokens = token_splits["val_monitor"]
    val_ablation_tokens = token_splits["val_ablation"]
    test_replication_tokens = token_splits["test_replication"]

    train_dataset = PackedTokenDataset(train_tokens, cfg.max_seq_len)
    val_monitor_dataset = PackedTokenDataset(val_monitor_tokens, cfg.max_seq_len)
    val_ablation_dataset = PackedTokenDataset(val_ablation_tokens, cfg.max_seq_len)
    test_replication_dataset = PackedTokenDataset(test_replication_tokens, cfg.max_seq_len)

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=True,
    )
    val_monitor_loader = DataLoader(
        val_monitor_dataset,
        batch_size=args.eval_batch_size or args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )
    val_ablation_loader = DataLoader(
        val_ablation_dataset,
        batch_size=args.eval_batch_size or args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )
    test_replication_loader = DataLoader(
        test_replication_dataset,
        batch_size=args.eval_batch_size or args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )

    model = ToyMamba2(cfg).to(device)
    n_params = model.num_parameters()
    print(f"Model parameters: {n_params:,} ({n_params / 1e6:.2f}M)")
    print(
        f"Train samples: {len(train_dataset):,} | "
        f"Val-monitor samples: {len(val_monitor_dataset):,} | "
        f"Val-ablation samples: {len(val_ablation_dataset):,} | "
        f"Test-replication samples: {len(test_replication_dataset):,}"
    )
    print(f"Split info: {split_info}")
    print(f"Split checks: {split_checks}")

    optimizer = make_optimizer(model, lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda s: cosine_with_warmup(s, args.warmup_steps, args.max_steps),
    )

    use_scaler = device.type == "cuda" and amp_dtype == torch.float16
    scaler = torch.amp.GradScaler(device.type, enabled=use_scaler)

    effective_batch = args.batch_size * args.grad_accum
    tokens_per_optimizer_step = effective_batch * cfg.max_seq_len
    flops_info = estimate_mamba2_flops_per_optimizer_step(cfg, effective_batch)

    run_name = f"toy_mamba2_{args.model_size}_{args.dataset}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    log_path = Path(args.log_dir) / f"{run_name}.jsonl"
    checkpoint_dir = Path(args.save_dir) / run_name

    run_config = {
        "model_config": asdict(cfg),
        "n_params": n_params,
        "dataset": args.dataset,
        "split_info": split_info,
        "split_checks": split_checks,
        "batch_size": args.batch_size,
        "grad_accum": args.grad_accum,
        "effective_batch": effective_batch,
        "tokens_per_optimizer_step": tokens_per_optimizer_step,
        "max_steps": args.max_steps,
        "lr": args.lr,
        "weight_decay": args.weight_decay,
        "warmup_steps": args.warmup_steps,
        "estimated_flops": flops_info,
        "device": str(device),
        "amp_dtype": str(amp_dtype),
        "use_amp": use_amp,
        "early_stopping_patience": args.early_stopping_patience,
        "early_stopping_min_delta": args.early_stopping_min_delta,
    }
    if device.type == "cuda":
        run_config["gpu_name"] = torch.cuda.get_device_name(device)
        run_config["gpu_memory_gb"] = round(torch.cuda.get_device_properties(device).total_memory / 1e9, 1)

    logger = TrainingLogger(str(log_path), run_config)
    print(f"Logging to: {log_path}")

    early = EarlyStoppingState(
        patience=args.early_stopping_patience,
        min_delta=args.early_stopping_min_delta,
        warmup_evals=args.early_stopping_warmup_evals,
    )
    recent_val_losses: list[float] = []

    model.train()
    optimizer.zero_grad(set_to_none=True)

    micro_step = 0
    optimizer_step = 0
    tokens_seen = 0
    running_loss = 0.0
    running_micro_batches = 0
    cumulative_estimated_flops = 0.0
    t_start = time.time()

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    print("\nStarting training")
    print(f"Max optimizer steps: {args.max_steps:,}")
    print(f"Validation every: {args.eval_every:,} optimizer steps")
    print(f"Early stopping patience: {args.early_stopping_patience} evals")
    print("\n")

    should_stop = False

    while optimizer_step < args.max_steps and not should_stop:
        for x, y in train_loader:
            if optimizer_step >= args.max_steps or should_stop:
                break

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            micro_step += 1
            tokens_seen += x.numel()
            step_t0 = time.time()

            with torch.amp.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
                _, loss = model(x, y)
                loss_for_backward = loss / args.grad_accum

            scaler.scale(loss_for_backward).backward()

            running_loss += float(loss.item())
            running_micro_batches += 1

            did_optimizer_step = (micro_step % args.grad_accum == 0)
            if did_optimizer_step:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip).item()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

                optimizer_step += 1
                cumulative_estimated_flops += flops_info["estimated_flops_per_optimizer_step"]

                if optimizer_step % args.log_every == 0:
                    elapsed = time.time() - t_start
                    avg_train_loss = running_loss / max(running_micro_batches, 1)
                    train_ppl = math.exp(min(avg_train_loss, 20))
                    tok_per_sec = tokens_seen / max(elapsed, 1e-8)
                    lr_now = scheduler.get_last_lr()[0]
                    estimated_tflops = cumulative_estimated_flops / max(elapsed, 1e-8) / 1e12

                    metrics = {
                        "optimizer_step": optimizer_step,
                        "micro_step": micro_step,
                        "train_loss": round(avg_train_loss, 5),
                        "train_ppl": round(train_ppl, 3),
                        "lr": lr_now,
                        "grad_norm": round(grad_norm, 4),
                        "tokens_seen": tokens_seen,
                        "tokens_per_sec": round(tok_per_sec),
                        "step_time_ms": round((time.time() - step_t0) * 1000, 1),
                        "elapsed_min": round(elapsed / 60, 2),
                        "estimated_tflops": round(estimated_tflops, 2),
                        "dataset_epochs_by_tokens": round(tokens_seen / len(train_tokens), 4),
                        **gpu_stats(device),
                    }
                    logger.log_step(metrics)

                    print(
                        f"step {optimizer_step:>6d}/{args.max_steps} | "
                        f"train_loss {avg_train_loss:.4f} | "
                        f"train_ppl {train_ppl:.2f} | "
                        f"lr {lr_now:.2e} | "
                        f"tok/s {tok_per_sec:,.0f} | "
                        f"grad {grad_norm:.3f}"
                    )

                    running_loss = 0.0
                    running_micro_batches = 0

                if optimizer_step % args.eval_every == 0 or optimizer_step == 1:
                    eval_metrics = evaluate(
                        model=model,
                        loader=val_monitor_loader,
                        device=device,
                        amp_dtype=amp_dtype,
                        max_batches=args.eval_batches,
                        use_amp=use_amp,
                    )
                    val_loss_float = float(eval_metrics["val_loss"])
                    recent_val_losses.append(val_loss_float)
                    recent_val_losses = recent_val_losses[-args.plateau_window :]

                    improved, should_stop = early.update(val_loss_float, optimizer_step)
                    training_status = is_model_trained_enough(
                        current_val_loss=val_loss_float,
                        best_val_loss=early.best_val_loss,
                        recent_val_losses=recent_val_losses,
                        min_relative_improvement=args.plateau_min_relative_improvement,
                    )

                    eval_payload = {
                        "optimizer_step": optimizer_step,
                        "micro_step": micro_step,
                        **eval_metrics,
                        "best_val_loss": round(early.best_val_loss, 5),
                        "best_step": early.best_step,
                        "bad_evals": early.bad_evals,
                        "improved": improved,
                        "training_status": training_status,
                    }
                    logger.log_eval(eval_payload)

                    print(
                        f"EVAL step {optimizer_step:>6d} | "
                        f"val_loss {eval_metrics['val_loss']:.5f} | "
                        f"val_ppl {eval_metrics['val_ppl']:.2f} | "
                        f"best {early.best_val_loss:.5f} @ step {early.best_step} | "
                        f"status {training_status['status']}"
                    )

                    if improved:
                        save_checkpoint(
                            checkpoint_dir / "best.pt",
                            model,
                            optimizer,
                            scheduler,
                            cfg,
                            optimizer_step,
                            micro_step,
                            early.best_val_loss,
                            extra={"eval_metrics": eval_payload},
                        )

                    if should_stop:
                        print(
                            "Early stopping triggered: validation loss did not improve "
                            f"by at least {args.early_stopping_min_delta} for "
                            f"{args.early_stopping_patience} evals."
                        )
                        break

                if args.save_every > 0 and optimizer_step % args.save_every == 0:
                    save_checkpoint(
                        checkpoint_dir / f"step_{optimizer_step}.pt",
                        model,
                        optimizer,
                        scheduler,
                        cfg,
                        optimizer_step,
                        micro_step,
                        early.best_val_loss,
                    )

        # If the loader is exhausted before max_steps, start another epoch automatically.

    total_time = time.time() - t_start
    # Final held-out evaluations.
    # val_ablation can be used to compare variants.
    # test_replication should be used only for final reporting.
    ablation_metrics = evaluate(
        model=model,
        loader=val_ablation_loader,
        device=device,
        amp_dtype=amp_dtype,
        max_batches=args.final_eval_batches,
        use_amp=use_amp,
    )
    test_replication_metrics = evaluate(
        model=model,
        loader=test_replication_loader,
        device=device,
        amp_dtype=amp_dtype,
        max_batches=args.final_eval_batches,
        use_amp=use_amp,
    )

    final_metrics = {
        "optimizer_steps": optimizer_step,
        "micro_steps": micro_step,
        "total_time_sec": round(total_time, 1),
        "total_time_hours": round(total_time / 3600, 3),
        "tokens_seen": tokens_seen,
        "avg_tokens_per_sec": round(tokens_seen / max(total_time, 1e-8)),
        "best_val_loss": round(early.best_val_loss, 5),
        "best_step": early.best_step,
        "stopped_by_early_stopping": should_stop,
        "val_ablation_metrics": ablation_metrics,
        "test_replication_metrics": test_replication_metrics,
        **gpu_stats(device),
    }
    logger.log_final(final_metrics)
    logger.close()

    save_checkpoint(
        checkpoint_dir / "last.pt",
        model,
        optimizer,
        scheduler,
        cfg,
        optimizer_step,
        micro_step,
        early.best_val_loss,
    )

    print("\nTraining complete")
    print(json.dumps(final_metrics, indent=2))

    # Quick qualitative sample from the final model.
    model.eval()
    prompt = "Once upon a time"
    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    out_ids = model.generate(
        prompt_ids,
        max_new_tokens=100,
        temperature=0.8,
        top_k=40,
        use_recurrent=(args.sample_generation_mode == "recurrent"),
        optimized_prefill=(not args.no_optimized_prefill),
    )
    print("\nSample generation:")
    print(tokenizer.decode(out_ids[0].tolist()))

In [ ]:
def make_colab_config() -> SimpleNamespace:

    # Model and data
    MODEL_SIZE = "half"             # "quarter", "half", or "full"
    DATASET = "tinystories"         # "tinystories", "wikitext-2", or "file:/path/to/text.txt"
    MAX_TOKENS = None

    # Held-out split fractions. For TinyStories/file datasets, these are carved from the tail of the loaded token stream.
    VAL_MONITOR_FRACTION = 0.02       # early stopping / checkpoint selection
    VAL_ABLATION_FRACTION = 0.015     # variant comparison
    TEST_REPLICATION_FRACTION = 0.015 # final report only

    VAL_MONITOR_MAX_TOKENS = 2_000_000
    VAL_ABLATION_MAX_TOKENS = 1_000_000
    TEST_REPLICATION_MAX_TOKENS = 1_000_000

    # Sequence and batch settings
    SEQ_LEN = 1024
    BLOCK_LEN = 64
    BATCH_SIZE = 24
    EVAL_BATCH_SIZE = None           # None means use BATCH_SIZE
    GRAD_ACCUM = 1
    NUM_WORKERS = 2

    # Optimization
    LR = 3e-4
    WEIGHT_DECAY = 0.01
    DROPOUT = 0.0
    WARMUP_STEPS = 1000
    MAX_STEPS = 100_000               # safety cap. early stopping can stop earlier
    GRAD_CLIP = 1.0

    # Logging / eval / stopping
    LOG_EVERY = 100
    EVAL_EVERY = 500
    EVAL_BATCHES = 50
    FINAL_EVAL_BATCHES = 200
    SAVE_EVERY = 5000
    SAVE_DIR = "/content/drive/MyDrive/models/mamba2_toy/checkpoints"
    LOG_DIR = "/content/drive/MyDrive/models/mamba2_toy/logs"

    EARLY_STOPPING_PATIENCE = 8
    EARLY_STOPPING_MIN_DELTA = 0.002
    EARLY_STOPPING_WARMUP_EVALS = 3
    PLATEAU_WINDOW = 5
    PLATEAU_MIN_RELATIVE_IMPROVEMENT = 0.002

    # Correctness tests and generation
    TEST_SSD = True
    TEST_RECURRENT = True
    SAMPLE_GENERATION_MODE = "recurrent"  # "recurrent" or "full_prefix"
    NO_OPTIMIZED_PREFILL = False

    return SimpleNamespace(
        model_size=MODEL_SIZE,
        dataset=DATASET,
        max_tokens=MAX_TOKENS,
        val_monitor_fraction=VAL_MONITOR_FRACTION,
        val_ablation_fraction=VAL_ABLATION_FRACTION,
        test_replication_fraction=TEST_REPLICATION_FRACTION,
        val_monitor_max_tokens=VAL_MONITOR_MAX_TOKENS,
        val_ablation_max_tokens=VAL_ABLATION_MAX_TOKENS,
        test_replication_max_tokens=TEST_REPLICATION_MAX_TOKENS,
        seq_len=SEQ_LEN,
        block_len=BLOCK_LEN,
        batch_size=BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        grad_accum=GRAD_ACCUM,
        num_workers=NUM_WORKERS,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        dropout=DROPOUT,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        grad_clip=GRAD_CLIP,
        log_every=LOG_EVERY,
        eval_every=EVAL_EVERY,
        eval_batches=EVAL_BATCHES,
        final_eval_batches=FINAL_EVAL_BATCHES,
        save_every=SAVE_EVERY,
        save_dir=SAVE_DIR,
        log_dir=LOG_DIR,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
        early_stopping_warmup_evals=EARLY_STOPPING_WARMUP_EVALS,
        plateau_window=PLATEAU_WINDOW,
        plateau_min_relative_improvement=PLATEAU_MIN_RELATIVE_IMPROVEMENT,
        test_ssd=TEST_SSD,
        test_recurrent=TEST_RECURRENT,
        sample_generation_mode=SAMPLE_GENERATION_MODE,
        no_optimized_prefill=NO_OPTIMIZED_PREFILL,
    )


In [ ]:
train(make_colab_config())


Device: cuda | amp_dtype: torch.bfloat16 | use_amp: True
SSD correctness test passed.
Recurrent step-by-step prefill matches full forward test passed.
Optimized parallel prefill cache test passed.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Model parameters: 26,852,384 (26.85M)
Train samples: 456,906 | Val-monitor samples: 1,953 | Val-ablation samples: 976 | Test-replication samples: 976
Split info: {'dataset': 'tinystories', 'split_policy': 'tail holdout from train token stream', 'fractions': {'val_monitor': 0.02, 'val_ablation': 0.015, 'test_replication': 0.015}, 'n_total_loaded_tokens': 471872517, 'n_tokens': {'train': 467872517, 'val_monitor': 2000000, 'val_ablation': 1000000, 'test_replication': 1000000}, 'heldout_names': ['val_monitor', 'val_ablation', 'test_replication']}
Split checks: {'train': {'tokens': 467872517, 'samples': 456906}, 'val_monitor': {'tokens': 2000000, 'samples': 1953}, 'val_ablation': {'tokens': 1000000, 'samples': 976}, 'test_replication': {'tokens': 1000000, 'samples': 976}, 'windows_cross_boundary': False, 'note': 'PackedTokenDataset is built separately for each split, so windows do not cross boundaries. This does not prove semantic deduplication.'}
Logging to: /content/drive/MyDrive/models/m